<a href="https://colab.research.google.com/github/Le2se0hy/XAI_An/blob/main/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 서울 아파트 XGBoost 회귀분석
#
# 분석:
# 1. 2022년 별도 분석
# 2. 2023년 별도 분석
# 3. 2022~2023년 전체 기간 분석
#
# 데이터 분할:
# - Train 70%
# - Test 30%
#
# 종속변수:
# - Log_Price_per_m2
#
# 주의:
# - Log_Price_per_m2는 설명변수에 포함하지 않음
# - Year_Sold와 Month_Sold도 설명변수에 포함하지 않음
# - Month_Sold는 계절 더미 생성에만 사용
# - heating이 정확히 '도시가스'인 경우만 1
# ============================================================


# ============================================================
# 0. 라이브러리 설치
# ============================================================
!pip -q install xgboost openpyxl


# ============================================================
# 1. 라이브러리
# ============================================================
import os
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor


# ============================================================
# 2. 사용자 설정
# ============================================================
FILE_PATH = "/content/drive/MyDrive/Seoul_Aprtment_FINAL.xlsx"

OUTPUT_PATH = (
    "/content/drive/MyDrive/"
    "Seoul_Apartment_XGB_Results.xlsx"
)

TARGET_COL = "Log_Price_per_m2"

TARGET_YEARS = [2022, 2023]

RANDOM_STATE = 42
TEST_SIZE = 0.30


# ============================================================
# 3. Google Drive 연결 및 데이터 불러오기
# ============================================================
drive.mount(
    "/content/drive",
    force_remount=True
)

df_raw = pd.read_excel(FILE_PATH)

print("=" * 80)
print("원본 데이터 shape:", df_raw.shape)
print("원본 컬럼 수:", len(df_raw.columns))
print("=" * 80)


# ============================================================
# 4. 숫자형 정리 함수
# ============================================================
def clean_numeric_series(series):
    """
    쉼표, 공백 등이 포함된 값을 숫자형으로 변환한다.
    숫자로 변환할 수 없는 값은 결측치로 처리한다.
    """

    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .replace({
            "-": np.nan,
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "null": np.nan,
            "NULL": np.nan
        }),
        errors="coerce"
    )


# ============================================================
# 5. 데이터 전처리 함수
# ============================================================
def prepare_seoul_df(df_input):
    df = df_input.copy()

    # --------------------------------------------------------
    # 숫자형 변수
    # --------------------------------------------------------
    numeric_cols = [
        "Log_Price_per_m2",

        "Size_m2",
        "Floor",
        "num_of_people",
        "Parking_per_Household",
        "Construction_Year",
        "max_floor",

        "Log_Dist_Subway",
        "Log_Dist_Green",
        "Log_Dist_Water",
        "Dist_CBD",
        "Bus_Stop",
        "High_School_Count",

        "Population",
        "Sex_ratio",
        "Pop. Density",
        "Median age Population",
        "Young Population",

        "Year_Sold",
        "Month_Sold"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = clean_numeric_series(df[col])

    # --------------------------------------------------------
    # 난방방식 더미
    # 정확히 '도시가스'이면 1
    # 나머지는 모두 0
    # --------------------------------------------------------
    if "heating" not in df.columns:
        raise ValueError(
            "데이터에 'heating' 변수가 없습니다."
        )

    heating_clean = (
        df["heating"]
        .astype("string")
        .str.strip()
        .fillna("")
    )

    df["Heating_Dummy"] = (
        heating_clean
        .eq("도시가스")
        .astype(int)
    )

    # --------------------------------------------------------
    # 계절 더미
    # 여름 6·7·8월은 기준범주
    # --------------------------------------------------------
    if "Month_Sold" not in df.columns:
        raise ValueError(
            "데이터에 'Month_Sold' 변수가 없습니다."
        )

    month = df["Month_Sold"]

    df["Spring"] = (
        month.isin([3, 4, 5])
        .astype(int)
    )

    df["Fall"] = (
        month.isin([9, 10, 11])
        .astype(int)
    )

    df["Winter"] = (
        month.isin([12, 1, 2])
        .astype(int)
    )

    return df


# ============================================================
# 6. 최종 설명변수
# ============================================================
FEATURE_COLS = [
    # --------------------------------------------------------
    # 개별 주택 및 단지 특성
    # --------------------------------------------------------
    "Size_m2",
    "Floor",
    "num_of_people",
    "Parking_per_Household",
    "Construction_Year",
    "max_floor",
    "Heating_Dummy",

    # --------------------------------------------------------
    # 거리 및 주변 시설
    # --------------------------------------------------------
    "Log_Dist_Subway",
    "Log_Dist_Green",
    "Log_Dist_Water",
    "Dist_CBD",
    "Bus_Stop",
    "High_School_Count",

    # --------------------------------------------------------
    # 지역 인구 특성
    # 파생 복사 변수가 아닌 원래 변수 사용
    # --------------------------------------------------------
    "Population",
    "Sex_ratio",
    "Pop. Density",
    "Median age Population",
    "Young Population",

    # --------------------------------------------------------
    # 계절 통제변수
    # 여름이 기준범주
    # --------------------------------------------------------
    "Spring",
    "Fall",
    "Winter"
]


# ============================================================
# 7. XGBoost 실행 함수
# ============================================================
def run_xgboost_prediction(
    df_sub,
    feature_cols,
    target_col="Log_Price_per_m2",
    test_size=0.30,
    random_state=42
):
    """
    XGBoost 회귀모형

    - 지정한 모든 설명변수를 동일하게 사용
    - Train 70%, Test 30%
    - Train/Test R2와 RMSE 계산
    - Test 예측값 저장
    - 변수 중요도 저장
    """

    df_model = df_sub.copy()

    # 원래 데이터 행 번호 보관
    df_model["_Original_Index"] = df_model.index

    # --------------------------------------------------------
    # 변수 존재 여부 확인
    # 없는 변수를 조용히 제외하지 않고 오류 발생
    # --------------------------------------------------------
    required_cols = (
        feature_cols
        + [target_col]
    )

    missing_cols = [
        col
        for col in required_cols
        if col not in df_model.columns
    ]

    if missing_cols:
        raise ValueError(
            "다음 변수가 데이터에 없습니다: "
            f"{missing_cols}"
        )

    # --------------------------------------------------------
    # 설명변수와 종속변수 숫자형 변환
    # --------------------------------------------------------
    for col in required_cols:
        df_model[col] = clean_numeric_series(
            df_model[col]
        )

    # 무한대는 결측치로 처리
    df_model = df_model.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # --------------------------------------------------------
    # 변수별 결측치 확인
    # --------------------------------------------------------
    missing_count = (
        df_model[required_cols]
        .isna()
        .sum()
        .sort_values(ascending=False)
    )

    print("\n결측치 상위 변수")
    print(missing_count.head(10))

    # --------------------------------------------------------
    # 분석 변수에 결측치가 있는 행 제거
    # --------------------------------------------------------
    df_model = (
        df_model
        .dropna(subset=required_cols)
        .reset_index(drop=True)
    )

    if len(df_model) < 30:
        raise ValueError(
            "결측치 제거 후 유효 표본이 부족합니다. "
            f"유효 표본 수: {len(df_model)}"
        )

    # --------------------------------------------------------
    # X와 y 구성
    # 종속변수는 X에 포함되지 않음
    # --------------------------------------------------------
    X = df_model[feature_cols].copy()
    y = df_model[target_col].copy()

    original_index = df_model[
        "_Original_Index"
    ].copy()

    # --------------------------------------------------------
    # Train 70%, Test 30%
    # 인덱스도 함께 분할
    # --------------------------------------------------------
    (
        X_train,
        X_test,
        y_train,
        y_test,
        index_train,
        index_test
    ) = train_test_split(
        X,
        y,
        original_index,
        test_size=test_size,
        random_state=random_state
    )

    # --------------------------------------------------------
    # XGBoost 모델
    # 기존 첫 번째 코드의 설정 유지
    # --------------------------------------------------------
    xgb_model = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1
    )

    # 모델 학습
    xgb_model.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # Train 예측 및 성능
    # --------------------------------------------------------
    y_pred_train = xgb_model.predict(
        X_train
    )

    train_r2 = r2_score(
        y_train,
        y_pred_train
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            y_pred_train
        )
    )

    # --------------------------------------------------------
    # Test 예측 및 성능
    # --------------------------------------------------------
    y_pred_test = xgb_model.predict(
        X_test
    )

    test_r2 = r2_score(
        y_test,
        y_pred_test
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred_test
        )
    )

    # --------------------------------------------------------
    # Train 예측값표
    # --------------------------------------------------------
    train_prediction_df = pd.DataFrame({
        "Original_Index": index_train.to_numpy(),
        "Dataset": "Train",
        "Actual": y_train.to_numpy(),
        "Predicted": y_pred_train,
        "Residual": (
            y_train.to_numpy()
            - y_pred_train
        )
    }).reset_index(drop=True)

    # --------------------------------------------------------
    # Test 예측값표
    # --------------------------------------------------------
    test_prediction_df = pd.DataFrame({
        "Original_Index": index_test.to_numpy(),
        "Dataset": "Test",
        "Actual": y_test.to_numpy(),
        "Predicted": y_pred_test,
        "Residual": (
            y_test.to_numpy()
            - y_pred_test
        )
    }).reset_index(drop=True)

    # --------------------------------------------------------
    # 변수 중요도
    # --------------------------------------------------------
    importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": (
            xgb_model.feature_importances_
        )
    }).sort_values(
        by="Importance",
        ascending=False
    ).reset_index(drop=True)

    return {
        "model": xgb_model,

        "train_r2": train_r2,
        "train_rmse": train_rmse,

        "test_r2": test_r2,
        "test_rmse": test_rmse,

        "train_prediction_df": (
            train_prediction_df
        ),

        "test_prediction_df": (
            test_prediction_df
        ),

        "importance_df": importance_df,

        "n_total": len(df_model),
        "n_train": len(X_train),
        "n_test": len(X_test),

        "used_features": feature_cols
    }


# ============================================================
# 8. 데이터 전처리
# ============================================================
df_all = prepare_seoul_df(df_raw)


# ============================================================
# 9. 최종 변수 확인
# ============================================================
missing_features = [
    col
    for col in FEATURE_COLS + [TARGET_COL]
    if col not in df_all.columns
]

if missing_features:
    raise ValueError(
        "분석에 필요한 변수가 누락되었습니다: "
        f"{missing_features}"
    )

print("\n최종 설명변수 수:", len(FEATURE_COLS))

for number, feature in enumerate(
    FEATURE_COLS,
    start=1
):
    print(f"{number:02d}. {feature}")


# ============================================================
# 10. 연도별 XGBoost 분석
# ============================================================
yearly_results = []
yearly_prediction_dict = {}
yearly_importance_dict = {}

for year in TARGET_YEARS:

    if "Year_Sold" not in df_all.columns:
        raise ValueError(
            "데이터에 'Year_Sold' 변수가 없습니다."
        )

    df_year = df_all[
        df_all["Year_Sold"] == year
    ].copy()

    print("\n")
    print("=" * 80)
    print(f"{year}년 XGBoost 분석")
    print("=" * 80)

    print(
        f"결측 제거 전 표본 수: "
        f"{len(df_year):,}"
    )

    if len(df_year) < 30:
        print(
            f"{year}년 표본 수가 부족하여 "
            "분석에서 제외합니다."
        )
        continue

    result = run_xgboost_prediction(
        df_sub=df_year,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    yearly_results.append({
        "Year": year,

        "Total_N": result["n_total"],
        "Train_N": result["n_train"],
        "Test_N": result["n_test"],

        "Train_R2": result["train_r2"],
        "Train_RMSE": result["train_rmse"],

        "Test_R2": result["test_r2"],
        "Test_RMSE": result["test_rmse"]
    })

    yearly_prediction_dict[year] = {
        "Train": result[
            "train_prediction_df"
        ],

        "Test": result[
            "test_prediction_df"
        ]
    }

    yearly_importance_dict[year] = (
        result["importance_df"]
    )

    print(f"\n[{year}년 사용 변수]")
    print(result["used_features"])

    print(f"\n[{year}년 Train 성능]")
    print(
        f"R²   : "
        f"{result['train_r2']:.4f}"
    )
    print(
        f"RMSE : "
        f"{result['train_rmse']:.4f}"
    )

    print(f"\n[{year}년 Test 성능]")
    print(
        f"R²   : "
        f"{result['test_r2']:.4f}"
    )
    print(
        f"RMSE : "
        f"{result['test_rmse']:.4f}"
    )

    print(
        f"\nTrain N : "
        f"{result['n_train']:,}"
    )

    print(
        f"Test N  : "
        f"{result['n_test']:,}"
    )

    print("\n변수 중요도 상위 10개")
    display(
        result["importance_df"].head(10)
    )


# ============================================================
# 11. 전체 기간 XGBoost 분석
# ============================================================
print("\n")
print("=" * 80)
print("전체 기간 XGBoost 분석")
print("=" * 80)

full_result = run_xgboost_prediction(
    df_sub=df_all,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("\n[전체 기간 사용 변수]")
print(full_result["used_features"])

print("\n[전체 기간 Train 성능]")
print(
    f"R²   : "
    f"{full_result['train_r2']:.4f}"
)
print(
    f"RMSE : "
    f"{full_result['train_rmse']:.4f}"
)

print("\n[전체 기간 Test 성능]")
print(
    f"R²   : "
    f"{full_result['test_r2']:.4f}"
)
print(
    f"RMSE : "
    f"{full_result['test_rmse']:.4f}"
)

print(
    f"\nTrain N : "
    f"{full_result['n_train']:,}"
)

print(
    f"Test N  : "
    f"{full_result['n_test']:,}"
)

print("\n전체 기간 변수 중요도 상위 10개")
display(
    full_result["importance_df"].head(10)
)


# ============================================================
# 12. 성능 비교표 생성
# ============================================================
performance_table = pd.DataFrame(
    yearly_results
)

full_performance_row = pd.DataFrame([{
    "Year": "Full Sample",

    "Total_N": full_result["n_total"],
    "Train_N": full_result["n_train"],
    "Test_N": full_result["n_test"],

    "Train_R2": full_result["train_r2"],
    "Train_RMSE": full_result["train_rmse"],

    "Test_R2": full_result["test_r2"],
    "Test_RMSE": full_result["test_rmse"]
}])

performance_table = pd.concat(
    [
        performance_table,
        full_performance_row
    ],
    ignore_index=True
)

print("\n[XGBoost 성능 비교표]")
display(performance_table)


# ============================================================
# 13. 예측값 일부 확인
# ============================================================
for year in yearly_prediction_dict:

    print(
        f"\n[{year}년 Train 예측값 일부]"
    )

    display(
        yearly_prediction_dict[
            year
        ]["Train"].head()
    )

    print(
        f"\n[{year}년 Test 예측값 일부]"
    )

    display(
        yearly_prediction_dict[
            year
        ]["Test"].head()
    )

print("\n[전체 기간 Test 예측값 일부]")
display(
    full_result[
        "test_prediction_df"
    ].head()
)


# ============================================================
# 14. 엑셀 저장
# ============================================================
with pd.ExcelWriter(
    OUTPUT_PATH,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # 성능 비교표
    # --------------------------------------------------------
    performance_table.to_excel(
        writer,
        sheet_name="XGB_Performance",
        index=False
    )

    # --------------------------------------------------------
    # 사용 변수 목록
    # --------------------------------------------------------
    pd.DataFrame({
        "Feature": FEATURE_COLS
    }).to_excel(
        writer,
        sheet_name="Used_Features",
        index=False
    )

    # --------------------------------------------------------
    # 연도별 예측값 및 변수 중요도
    # --------------------------------------------------------
    for year in yearly_prediction_dict:

        yearly_prediction_dict[
            year
        ]["Train"].to_excel(
            writer,
            sheet_name=f"Train_Pred_{year}",
            index=False
        )

        yearly_prediction_dict[
            year
        ]["Test"].to_excel(
            writer,
            sheet_name=f"Test_Pred_{year}",
            index=False
        )

        yearly_importance_dict[
            year
        ].to_excel(
            writer,
            sheet_name=f"Importance_{year}",
            index=False
        )

    # --------------------------------------------------------
    # 전체 기간 예측값
    # --------------------------------------------------------
    full_result[
        "train_prediction_df"
    ].to_excel(
        writer,
        sheet_name="Train_Pred_Full",
        index=False
    )

    full_result[
        "test_prediction_df"
    ].to_excel(
        writer,
        sheet_name="Test_Pred_Full",
        index=False
    )

    # --------------------------------------------------------
    # 전체 기간 변수 중요도
    # --------------------------------------------------------
    full_result[
        "importance_df"
    ].to_excel(
        writer,
        sheet_name="Importance_Full",
        index=False
    )


# ============================================================
# 15. 저장 완료
# ============================================================
print("\n")
print("=" * 80)
print("XGBoost 분석 완료")
print("=" * 80)

print("\n저장 경로:")
print(OUTPUT_PATH)

print("\n설명변수 수:")
print(len(FEATURE_COLS))

print("\n설명변수:")
print(FEATURE_COLS)

Mounted at /content/drive
원본 데이터 shape: (162375, 41)
원본 컬럼 수: 41

최종 설명변수 수: 21
01. Size_m2
02. Floor
03. num_of_people
04. Parking_per_Household
05. Construction_Year
06. max_floor
07. Heating_Dummy
08. Log_Dist_Subway
09. Log_Dist_Green
10. Log_Dist_Water
11. Dist_CBD
12. Bus_Stop
13. High_School_Count
14. Population
15. Sex_ratio
16. Pop. Density
17. Median age Population
18. Young Population
19. Spring
20. Fall
21. Winter


2022년 XGBoost 분석
결측 제거 전 표본 수: 10,579

결측치 상위 변수
Size_m2                  0
Floor                    0
num_of_people            0
Parking_per_Household    0
Construction_Year        0
max_floor                0
Heating_Dummy            0
Log_Dist_Subway          0
Log_Dist_Green           0
Log_Dist_Water           0
dtype: int64

[2022년 사용 변수]
['Size_m2', 'Floor', 'num_of_people', 'Parking_per_Household', 'Construction_Year', 'max_floor', 'Heating_Dummy', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD', 'Bus_Stop', 'High_School_Count', 'Popula

,Feature,Importance
0,Heating_Dummy,0.177017
1,Young Population,0.157301
2,num_of_people,0.100633
3,Construction_Year,0.068310
4,max_floor,0.066242
5,Population,0.054469
6,Fall,0.040584
7,Sex_ratio,0.038268
8,Bus_Stop,0.036444
9,Median age Population,0.032905




2023년 XGBoost 분석
결측 제거 전 표본 수: 31,071

결측치 상위 변수
Size_m2                  0
Floor                    0
num_of_people            0
Parking_per_Household    0
Construction_Year        0
max_floor                0
Heating_Dummy            0
Log_Dist_Subway          0
Log_Dist_Green           0
Log_Dist_Water           0
dtype: int64

[2023년 사용 변수]
['Size_m2', 'Floor', 'num_of_people', 'Parking_per_Household', 'Construction_Year', 'max_floor', 'Heating_Dummy', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD', 'Bus_Stop', 'High_School_Count', 'Population', 'Sex_ratio', 'Pop. Density', 'Median age Population', 'Young Population', 'Spring', 'Fall', 'Winter']

[2023년 Train 성능]
R²   : 0.9587
RMSE : 0.0878

[2023년 Test 성능]
R²   : 0.9380
RMSE : 0.1080

Train N : 21,749
Test N  : 9,322

변수 중요도 상위 10개


,Feature,Importance
0,Heating_Dummy,0.337757
1,Young Population,0.171999
2,Construction_Year,0.056712
3,Population,0.044393
4,num_of_people,0.043990
5,max_floor,0.039622
6,Sex_ratio,0.038359
7,Median age Population,0.036553
8,Bus_Stop,0.031381
9,High_School_Count,0.031128




전체 기간 XGBoost 분석

결측치 상위 변수
Size_m2                  0
Floor                    0
num_of_people            0
Parking_per_Household    0
Construction_Year        0
max_floor                0
Heating_Dummy            0
Log_Dist_Subway          0
Log_Dist_Green           0
Log_Dist_Water           0
dtype: int64

[전체 기간 사용 변수]
['Size_m2', 'Floor', 'num_of_people', 'Parking_per_Household', 'Construction_Year', 'max_floor', 'Heating_Dummy', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD', 'Bus_Stop', 'High_School_Count', 'Population', 'Sex_ratio', 'Pop. Density', 'Median age Population', 'Young Population', 'Spring', 'Fall', 'Winter']

[전체 기간 Train 성능]
R²   : 0.9317
RMSE : 0.1210

[전체 기간 Test 성능]
R²   : 0.9235
RMSE : 0.1273

Train N : 113,662
Test N  : 48,713

전체 기간 변수 중요도 상위 10개


...


[2023년 Test 예측값 일부]


,Original_Index,Dataset,Actual,Predicted,Residual
0,113125,Test,16.454920,16.400206,0.054714
1,48180,Test,16.102828,16.196194,-0.093366
2,113290,Test,16.659107,16.659115,-0.000007
3,48150,Test,16.075036,16.071213,0.003823
4,59953,Test,16.046305,16.012854,0.033451



[전체 기간 Test 예측값 일부]


,Original_Index,Dataset,Actual,Predicted,Residual
0,62192,Test,16.463642,16.465666,-0.002024
1,63454,Test,16.152554,16.235413,-0.082859
2,94637,Test,16.483932,16.036381,0.447551
3,68478,Test,16.355151,16.467573,-0.112422
4,14776,Test,16.445963,16.547968,-0.102005




XGBoost 분석 완료

저장 경로:
/content/drive/MyDrive/Seoul_Apartment_XGB_Results.xlsx

설명변수 수:
21

설명변수:
['Size_m2', 'Floor', 'num_of_people', 'Parking_per_Household', 'Construction_Year', 'max_floor', 'Heating_Dummy', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD', 'Bus_Stop', 'High_School_Count', 'Population', 'Sex_ratio', 'Pop. Density', 'Median age Population', 'Young Population', 'Spring', 'Fall', 'Winter']


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from google.colab import drive

# ============================================================
# 0. XGBoost 설치
# Colab에서 설치되지 않은 경우에만 주석 해제
# ============================================================
# !pip install xgboost openpyxl -q


# ============================================================
# 1. 데이터 로드
# ============================================================
drive.mount("/content/drive")

file_path = "/content/drive/MyDrive/Seoul_Aprtment_FINAL.xlsx"
df_raw = pd.read_excel(file_path)


# ============================================================
# 2. 전처리 함수
# ============================================================
def prepare_seoul_df_final(df_input):
    df = df_input.copy()

    # --------------------------------------------------------
    # Size
    # --------------------------------------------------------
    if "Size_m2" in df.columns:
        df["Size_Level"] = pd.to_numeric(
            df["Size_m2"],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Population Density
    # --------------------------------------------------------
    if "Pop. Density" in df.columns:
        df["Pop_Density_Level"] = pd.to_numeric(
            df["Pop. Density"],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Units
    # --------------------------------------------------------
    if "num_of_people" in df.columns:
        df["Units_Level"] = pd.to_numeric(
            df["num_of_people"],
            errors="coerce"
        )

    # --------------------------------------------------------
    # 인구 관련 비율 변수
    # Median age, Young Population은
    # 이미 0.13 형태로 저장되어 있으므로 그대로 사용
    # --------------------------------------------------------
    ratio_map = {
        "Median age": "Medium_age_ratio",
        "Young Population": "Young_pop_ratio"
    }

    for original, new in ratio_map.items():
        if original in df.columns:
            df[new] = pd.to_numeric(
                df[original],
                errors="coerce"
            )

    # --------------------------------------------------------
    # 성비
    # 0~1 형태이면 그대로 사용하고,
    # 0~100 형태이면 100으로 나누어 사용
    # --------------------------------------------------------
    if "Sex_ratio" in df.columns:
        sex_ratio = pd.to_numeric(
            df["Sex_ratio"],
            errors="coerce"
        )

        valid_sex_ratio = sex_ratio.dropna()

        if (
            len(valid_sex_ratio) > 0
            and valid_sex_ratio.abs().max() > 2.0
        ):
            df["Sex_ratio_ratio"] = sex_ratio / 100.0
        else:
            df["Sex_ratio_ratio"] = sex_ratio

    # --------------------------------------------------------
    # CBD 거리: m → km
    # --------------------------------------------------------
    if "Dist_CBD" in df.columns:
        df["Dist_CBD_km"] = (
            pd.to_numeric(
                df["Dist_CBD"],
                errors="coerce"
            )
            / 1000.0
        )

    # --------------------------------------------------------
    # 계절 더미변수
    # 여름 6·7·8월이 기준범주
    # --------------------------------------------------------
    if "Month_Sold" in df.columns:
        month = pd.to_numeric(
            df["Month_Sold"],
            errors="coerce"
        )

        df["Spring"] = month.isin(
            [3, 4, 5]
        ).astype(int)

        df["Fall"] = month.isin(
            [9, 10, 11]
        ).astype(int)

        df["Winter"] = month.isin(
            [12, 1, 2]
        ).astype(int)

    return df


# ============================================================
# 3. XGBoost 실행 함수
# ============================================================
def run_xgboost_prediction(
    df_sub,
    feature_cols,
    target_col="Log_Price_per_m2",
    random_state=42
):
    """
    XGBoost 회귀모형

    - 데이터에 존재하는 설명변수만 사용
    - Train 70%, Test 30%
    - Train/Test R2 및 RMSE 계산
    - Test 예측값 저장
    - 변수 중요도 저장
    """

    df_model = df_sub.copy()

    # --------------------------------------------------------
    # 데이터에 실제로 존재하는 설명변수만 선택
    # 없는 설명변수는 조용히 제외
    # --------------------------------------------------------
    valid_features = [
        col for col in feature_cols
        if col in df_model.columns
    ]

    if target_col not in df_model.columns:
        raise ValueError(
            f"종속변수 '{target_col}'가 데이터에 없습니다."
        )

    if len(valid_features) == 0:
        raise ValueError(
            "사용할 수 있는 설명변수가 없습니다."
        )

    # --------------------------------------------------------
    # 사용 변수 숫자형 변환
    # --------------------------------------------------------
    for col in valid_features + [target_col]:
        df_model[col] = pd.to_numeric(
            df_model[col],
            errors="coerce"
        )

    # 무한대 값을 결측치로 처리
    df_model = df_model.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # --------------------------------------------------------
    # 사용 변수와 종속변수의 결측치 제거
    # --------------------------------------------------------
    df_model = (
        df_model
        .dropna(
            subset=valid_features + [target_col]
        )
        .reset_index(drop=True)
    )

    if len(df_model) < 30:
        raise ValueError(
            f"결측치 제거 후 유효 표본이 부족합니다: "
            f"{len(df_model)}개"
        )

    X = df_model[valid_features]
    y = df_model[target_col]

    # --------------------------------------------------------
    # Train 70%, Test 30%
    # --------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=random_state
    )

    # --------------------------------------------------------
    # XGBoost 모델
    # --------------------------------------------------------
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1
    )

    # 모델 학습
    xgb.fit(X_train, y_train)

    # --------------------------------------------------------
    # Train 성능
    # --------------------------------------------------------
    y_pred_train = xgb.predict(X_train)

    r2_train = r2_score(
        y_train,
        y_pred_train
    )

    rmse_train = np.sqrt(
        mean_squared_error(
            y_train,
            y_pred_train
        )
    )

    # --------------------------------------------------------
    # Test 성능
    # --------------------------------------------------------
    y_pred_test = xgb.predict(X_test)

    r2_test = r2_score(
        y_test,
        y_pred_test
    )

    rmse_test = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred_test
        )
    )

    # --------------------------------------------------------
    # Test 예측값
    # --------------------------------------------------------
    pred_df = pd.DataFrame({
        "Actual": y_test.to_numpy(),
        "Predicted": y_pred_test,
        "Residual": y_test.to_numpy() - y_pred_test
    }).reset_index(drop=True)

    # --------------------------------------------------------
    # 변수 중요도
    # --------------------------------------------------------
    importance_df = pd.DataFrame({
        "Feature": valid_features,
        "Importance": xgb.feature_importances_
    }).sort_values(
        by="Importance",
        ascending=False
    ).reset_index(drop=True)

    return {
        "model": xgb,
        "r2_train": r2_train,
        "rmse_train": rmse_train,
        "r2_test": r2_test,
        "rmse_test": rmse_test,
        "pred_df": pred_df,
        "importance_df": importance_df,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "used_features": valid_features
    }


# ============================================================
# 4. 데이터 준비
# ============================================================
df_all = prepare_seoul_df_final(df_raw)


# ============================================================
# 5. 사용할 설명변수
# ============================================================
features = [
     "Log_Price_per_m2",
        "Size_m2",
        "Floor",
        "num_of_people",
        "Parking_per_Household",
        "Construction_Year",
        "max_floor",
        "Log_Dist_Subway",
        "Log_Dist_Green",
        "Log_Dist_Water",
        "Dist_CBD",
        "Bus_Stop",
        "High_School_Count",
        "Population",
        "Sex_ratio",
        "Pop. Density",
        "Median age",
        "Young Population",
        "Year_Sold",
        "Month_Sold"
]

target = "Log_Price_per_m2"


# ============================================================
# 6. 연도별 XGBoost
# ============================================================
xgb_results_by_year = []
xgb_prediction_dict = {}
xgb_importance_dict = {}

for year in [2022, 2023]:

    if "Year_Sold" not in df_all.columns:
        raise ValueError(
            "데이터에 'Year_Sold' 변수가 없습니다."
        )

    year_values = pd.to_numeric(
        df_all["Year_Sold"],
        errors="coerce"
    )

    df_year = df_all[
        year_values == year
    ].copy()

    if len(df_year) < 30:
        print(
            f"{year}: 원자료 표본 수 부족으로 건너뜀"
        )
        continue

    try:
        xgb_out = run_xgboost_prediction(
            df_sub=df_year,
            feature_cols=features,
            target_col=target,
            random_state=42
        )

    except ValueError as error:
        print(f"{year}: {error}")
        continue

    xgb_results_by_year.append({
        "Year": year,
        "Train_R2": xgb_out["r2_train"],
        "Train_RMSE": xgb_out["rmse_train"],
        "Test_R2": xgb_out["r2_test"],
        "Test_RMSE": xgb_out["rmse_test"],
        "Train_N": xgb_out["n_train"],
        "Test_N": xgb_out["n_test"]
    })

    xgb_prediction_dict[year] = (
        xgb_out["pred_df"]
    )

    xgb_importance_dict[year] = (
        xgb_out["importance_df"]
    )

    print(f"\n[{year}년 XGBoost 결과]")

    print("\n사용 변수")
    print(xgb_out["used_features"])

    print("\nTrain 성능")
    print(
        f"R2   : {xgb_out['r2_train']:.4f}"
    )
    print(
        f"RMSE : {xgb_out['rmse_train']:.4f}"
    )

    print("\nTest 성능")
    print(
        f"R2   : {xgb_out['r2_test']:.4f}"
    )
    print(
        f"RMSE : {xgb_out['rmse_test']:.4f}"
    )

    print(
        f"\nTrain N : {xgb_out['n_train']:,}"
    )
    print(
        f"Test N  : {xgb_out['n_test']:,}"
    )

    print("\n변수 중요도 상위 10개")
    display(
        xgb_out["importance_df"].head(10)
    )


# ============================================================
# 7. 전체 기간 XGBoost
# ============================================================
xgb_full_out = run_xgboost_prediction(
    df_sub=df_all,
    feature_cols=features,
    target_col=target,
    random_state=42
)

print("\n[전체 기간 XGBoost 결과]")

print("\n사용 변수")
print(xgb_full_out["used_features"])

print("\nTrain 성능")
print(
    f"R2   : {xgb_full_out['r2_train']:.4f}"
)
print(
    f"RMSE : {xgb_full_out['rmse_train']:.4f}"
)

print("\nTest 성능")
print(
    f"R2   : {xgb_full_out['r2_test']:.4f}"
)
print(
    f"RMSE : {xgb_full_out['rmse_test']:.4f}"
)

print(
    f"\nTrain N : {xgb_full_out['n_train']:,}"
)
print(
    f"Test N  : {xgb_full_out['n_test']:,}"
)

print("\n전체 기간 변수 중요도 상위 10개")
display(
    xgb_full_out["importance_df"].head(10)
)


# ============================================================
# 8. 성능 비교표
# ============================================================
xgb_results_table = pd.DataFrame(
    xgb_results_by_year
)

full_row = pd.DataFrame([{
    "Year": "Full Sample",
    "Train_R2": xgb_full_out["r2_train"],
    "Train_RMSE": xgb_full_out["rmse_train"],
    "Test_R2": xgb_full_out["r2_test"],
    "Test_RMSE": xgb_full_out["rmse_test"],
    "Train_N": xgb_full_out["n_train"],
    "Test_N": xgb_full_out["n_test"]
}])

xgb_results_table = pd.concat(
    [xgb_results_table, full_row],
    ignore_index=True
)

print("\n[XGBoost 성능 비교표]")
display(xgb_results_table)


# ============================================================
# 9. 예측값 일부 확인
# ============================================================
for year, pred_df in xgb_prediction_dict.items():

    print(f"\n[{year}년 예측값 일부]")
    display(pred_df.head())

print("\n[전체 기간 예측값 일부]")
display(
    xgb_full_out["pred_df"].head()
)


# ============================================================
# 10. 엑셀 저장
# ============================================================
output_path = (
    "/content/drive/MyDrive/"
    "Seoul_Apartment_XGB_Results.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    # 성능 비교표
    xgb_results_table.to_excel(
        writer,
        sheet_name="XGB_Performance",
        index=False
    )

    # 연도별 예측값과 변수 중요도
    for year in xgb_prediction_dict:

        xgb_prediction_dict[year].to_excel(
            writer,
            sheet_name=f"Pred_{year}",
            index=False
        )

        xgb_importance_dict[year].to_excel(
            writer,
            sheet_name=f"Importance_{year}",
            index=False
        )

    # 전체 기간 예측값
    xgb_full_out["pred_df"].to_excel(
        writer,
        sheet_name="Pred_Full",
        index=False
    )

    # 전체 기간 변수 중요도
    xgb_full_out["importance_df"].to_excel(
        writer,
        sheet_name="Importance_Full",
        index=False
    )

print(f"\n저장 완료: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[2022년 XGBoost 결과]

사용 변수
['Log_Price_per_m2', 'Size_m2', 'Floor', 'num_of_people', 'Parking_per_Household', 'Construction_Year', 'max_floor', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD', 'Bus_Stop', 'High_School_Count', 'Population', 'Sex_ratio', 'Pop. Density', 'Young Population', 'Year_Sold', 'Month_Sold']

Train 성능
R2   : 0.9997
RMSE : 0.0082

Test 성능
R2   : 0.9987
RMSE : 0.0162

Train N : 7,405
Test N  : 3,174

변수 중요도 상위 10개


,Feature,Importance
0,Log_Price_per_m2,0.710929
1,Young Population,0.070725
2,num_of_people,0.042931
3,max_floor,0.028188
4,Population,0.027467
5,Construction_Year,0.027466
6,Bus_Stop,0.015285
7,Sex_ratio,0.013903
8,Pop. Density,0.012164
9,Parking_per_Household,0.011718



[2023년 XGBoost 결과]

사용 변수
['Log_Price_per_m2', 'Size_m2', 'Floor', 'num_of_people', 'Parking_per_Household', 'Construction_Year', 'max_floor', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD', 'Bus_Stop', 'High_School_Count', 'Population', 'Sex_ratio', 'Pop. Density', 'Young Population', 'Year_Sold', 'Month_Sold']

Train 성능
R2   : 0.9996
RMSE : 0.0082

Test 성능
R2   : 0.9987
RMSE : 0.0154

Train N : 21,749
Test N  : 9,322

변수 중요도 상위 10개


,Feature,Importance
0,Log_Price_per_m2,0.692587
1,Young Population,0.083099
2,max_floor,0.034072
3,Construction_Year,0.031124
4,Population,0.029130
5,num_of_people,0.026679
6,Bus_Stop,0.022004
7,Log_Dist_Subway,0.016496
8,High_School_Count,0.014769
9,Sex_ratio,0.013059



[전체 기간 XGBoost 결과]

사용 변수
['Log_Price_per_m2', 'Size_m2', 'Floor', 'num_of_people', 'Parking_per_Household', 'Construction_Year', 'max_floor', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD', 'Bus_Stop', 'High_School_Count', 'Population', 'Sex_ratio', 'Pop. Density', 'Young Population', 'Year_Sold', 'Month_Sold']

Train 성능
R2   : 0.9994
RMSE : 0.0111

Test 성능
R2   : 0.9993
RMSE : 0.0123

Train N : 113,662
Test N  : 48,713

전체 기간 변수 중요도 상위 10개


...


[2023년 예측값 일부]


,Actual,Predicted,Residual
0,16.454920,16.428175,0.026745
1,16.102828,16.100178,0.002650
2,16.659107,16.653408,0.005699
3,16.075036,16.079098,-0.004062
4,16.046305,16.035702,0.010603



[전체 기간 예측값 일부]


,Actual,Predicted,Residual
0,16.463642,16.463564,0.000078
1,16.152554,16.161175,-0.008621
2,16.483932,16.459156,0.024776
3,16.355151,16.359516,-0.004365
4,16.445963,16.455917,-0.009954



저장 완료: /content/drive/MyDrive/Seoul_Apartment_XGB_Results.xlsx
